# Notebook 01 - Exploratory Data Analysis

**Kelompok 08 - Teknik Komputer UI**

| NPM | Nama |
|-----|------|
| 2306242395 | Calvin Wirathama Katoroy |
| 2306220532 | Syahmi Hamdani |
| 2306220323 | Christover Angelo |
| 2306221024 | Matthew Immanuel |

**Tujuan:** Memahami distribusi kelas, statistik fitur, dan korelasi antar fitur pada dataset CIC-DDoS2019. Mengidentifikasi fitur pembeda traffic normal vs DDoS sebagai dasar pemilihan fitur untuk tahap preprocessing.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    # Already inside notebooks/ of the repo
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    # At repo root - step into notebooks/
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    # Not in repo - clone then navigate
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/tugas-akhir-ai')
    KAGGLE_DIR = DRIVE_ROOT / 'kaggle'
else:
    with open('../config.yaml') as f:
        import yaml; _cfg = yaml.safe_load(f)
    DRIVE_ROOT = Path(_cfg['data']['drive_root'])
    KAGGLE_DIR = DRIVE_ROOT / 'kaggle'

print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print(f'KAGGLE_DIR: {KAGGLE_DIR}')

In [ ]:
# Load all parquets
frames = []
parquet_files = sorted(KAGGLE_DIR.glob('*.parquet'))

if not parquet_files:
    raise FileNotFoundError(
        f'\nNo parquet files found in: {KAGGLE_DIR.resolve()}\n'
        f'Upload the Kaggle dataset parquet files there first.\n'
        f'Download from: https://www.kaggle.com/datasets/dhoogla/cicddos2019\n'
        f'Then upload all *.parquet files to: {KAGGLE_DIR.resolve()}'
    )

for p in parquet_files:
    df = pd.read_parquet(p)
    frames.append(df)
    print(f'{p.name}: {len(df):,} rows')

df = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(df):,} rows, {df.shape[1]} columns')

In [ ]:
# Normalize labels to binary
from src.preprocessing import normalize_labels, _LABEL_MAP

df = normalize_labels(df)
print(df['Label'].value_counts())

## Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Binary
counts = df['Label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#4C72B0', '#DD8452'])
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center')
axes[0].set_title('Binary Label Distribution')
axes[0].set_ylabel('Count')

# Multi-class (raw label col from original files)
# reload to get original labels
df_raw_labels = pd.concat([pd.read_parquet(p)['Label'].astype(str) for p in sorted(KAGGLE_DIR.glob('*.parquet'))], ignore_index=True)
orig_counts = df_raw_labels.value_counts()
orig_counts.plot(kind='barh', ax=axes[1])
axes[1].set_title('Per-Attack-Type Distribution')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

## Feature Statistics

In [ ]:
import yaml
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
features = cfg['features']

# Only keep features that exist in the dataframe
features = [f for f in features if f in df.columns]
print(f'Using {len(features)} features')
df[features].describe().T

In [ ]:
# Null/inf check
import numpy as np
numeric = df[features].select_dtypes(include=[np.number])
print('Null values:', numeric.isnull().sum().sum())
print('Inf values:', np.isinf(numeric.values).sum())

## Feature Distributions (normal vs ddos)

In [ ]:
n_cols = 3
n_rows = (len(features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, feat in enumerate(features):
    for label, grp in df.groupby('Label'):
        data = grp[feat].astype(float)
        # clip extreme outliers for readability
        lo, hi = data.quantile(0.01), data.quantile(0.99)
        data = data.clip(lo, hi)
        axes[i].hist(data, bins=50, alpha=0.6, label=label, density=True)
    axes[i].set_title(feat, fontsize=8)
    axes[i].legend(fontsize=7)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: normal vs ddos', y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Heatmap

In [ ]:
corr = df[features].astype(float).corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap.png', dpi=150)
plt.show()

# Flag highly correlated pairs
high_corr = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i, j]) > 0.95:
            high_corr.append((corr.index[i], corr.columns[j], round(corr.iloc[i,j], 3)))
if high_corr:
    print('Highly correlated pairs (|r| > 0.95):')
    for a, b, r in high_corr:
        print(f'  {a} <-> {b}: {r}')
else:
    print('No pairs with |r| > 0.95')